# **Initialization**

In [1]:
"""Start"""

'Start'

In [2]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import gc
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
import pulp
from ortools.linear_solver import pywraplp
from functools import lru_cache
# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Configuration & Data input**

In [3]:
# --- CONFIGURATION ---
# Path to your n20 folder containing .txt files
DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n50"
INPUT_CSV = "TSP_single_dual_bound_50_cus_results.csv"
OUTPUT_CSV = "result_of_ea_TSP_dual_bounds_50_cus.csv"
LOGS_DIR = "batch_logs"

if not os.path.exists(LOGS_DIR):
    os.makedirs(LOGS_DIR)

# --- GLOBAL VARIABLES (Initialize with Dummy Data) ---
# We create these so the functions in Cell 3 don't crash if checked early.
# These will be overwritten by the loop in Cell 4.
current_num_locations = 5
current_travel_cost = [[0.0]*5 for _ in range(5)]

print("✅ Globals initialized.")

# --- BATCH UTILITIES ---
def get_processed_instances(csv_path, logs_dir):
    """
    Returns a set of instances that exist in BOTH the CSV summary and the logs folder.
    """
    # 1. Get instances from CSV
    csv_instances = set()
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Instance' in df.columns:
                csv_instances = set(df['Instance'].unique())
        except:
            pass # CSV read failed, assume empty

    # 2. Get instances from Log Files
    log_instances = set()
    if os.path.exists(logs_dir):
        for filename in os.listdir(logs_dir):
            if filename.endswith("_log.txt"):
                # Extract "0.txt" from "0.txt_log.txt"
                instance_name = filename.replace("_log.txt", "")
                log_instances.add(instance_name)

    # 3. Return Intersection (Must be in BOTH to be considered "Done")
    return csv_instances.intersection(log_instances)

def append_result_to_csv(result_dict, csv_path):
    df = pd.DataFrame([result_dict])
    df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

print("✅ Configuration set.")

def read_tsp_cappart_format(file_path):
    with open(file_path, 'r') as f:
        values = f.read().split()
    iterator = iter(values)
    n = int(next(iterator))
    c = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(int(float(next(iterator))))
        c.append(row)
    return n, c

# Cell 3.5: Data Cleanup Utility

def clean_batch_data(csv_path, logs_dir):
    """
    Ensures consistency between the CSV summary and the Log files.
    1. Removes duplicate instances in CSV (keeps last).
    2. Removes CSV rows if the corresponding Log file is missing.
    3. Deletes Log files if the corresponding CSV row is missing.
    """
    print("🧹 Starting Data Cleanup...")
    
    # 1. Load CSV
    if not os.path.exists(csv_path):
        print("   -> CSV not found. Nothing to clean in CSV.")
        # If CSV missing but logs exist, we might want to clear logs, 
        # but usually better to leave them or delete manually to be safe.
        return 

    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        print("   -> CSV is empty.")
        return

    original_count = len(df)
    
    # 2. Deduplicate CSV (Keep the last run)
    df.drop_duplicates(subset=['Instance'], keep='last', inplace=True)
    dedup_count = len(df)
    if original_count > dedup_count:
        print(f"   -> Removed {original_count - dedup_count} duplicate rows from CSV.")

    # 3. Remove CSV rows without matching Log files
    valid_indices = []
    instances_in_csv = set()
    
    for index, row in df.iterrows():
        instance_name = row['Instance']
        expected_log = os.path.join(logs_dir, f"{instance_name}_log.txt")
        
        if os.path.exists(expected_log):
            valid_indices.append(index)
            instances_in_csv.add(instance_name)
        else:
            print(f"   -> Removing CSV row for '{instance_name}' (Log file missing).")
            
    # Filter dataframe to keep only valid rows
    df_clean = df.loc[valid_indices]
    
    # Save cleaned CSV
    df_clean.to_csv(csv_path, index=False)
    print(f"   -> CSV saved. Current number of row is: {len(df_clean)} (was {original_count}).")

    # 4. Remove Orphan Log files (Log exists, but not in CSV)
    if os.path.exists(logs_dir):
        files = os.listdir(logs_dir)
        for filename in files:
            if filename.endswith("_log.txt"):
                instance_from_log = filename.replace("_log.txt", "")
                
                if instance_from_log not in instances_in_csv:
                    file_path = os.path.join(logs_dir, filename)
                    try:
                        os.remove(file_path)
                        print(f"   -> Deleted orphan log: {filename} (Not in CSV).")
                    except OSError as e:
                        print(f"   -> Error deleting {filename}: {e}")

    print("✨ Data Cleanup Complete.\n")


✅ Globals initialized.
✅ Configuration set.


# **Model and dual bounds declaration**

In [4]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    n = current_num_locations
    c = current_travel_cost
    
    # 2. Initialize Model
    # Note: Ensure float_cost matches your data. Your snippet used False (Int), 
    # so we explicitly cast distances to Int in the reader.
    model = m_dp.Model(maximize=False, float_cost=True)

    customer = model.add_object_type(number=n)

    # 3. State Variables
    # U: Unvisited set (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    # i: Current location
    location = model.add_element_var(object_type=customer, target=0)

    # 4. Resource Tables
    travel_time = model.add_float_table(c)

    # 5. Transitions
    # Visit customer j
    for j in range(1, n):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
            ],
        )
        model.add_transition(visit)

    # Return to depot
    # Note: Removed 'time' effect from your snippet as it wasn't defined in the variables
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[
            (location, 0),
        ],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    # 8. Create Bundle (Model + Metadata)
    # This metadata dict allows your heuristics (like MST or assignment) 
    # to access the raw matrix data later.
    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
        # Add other keys if your dual bounds need them
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [5]:
def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    """
    1. Persistent 3-Index TSP Relaxation (MTZ Formulation).
    - Variables: x[i,j] (Flow), u[i] (Position/Potential).
    - Logic: Enforces connectivity via MTZ constraints.
    - Pattern: Cached internal worker '_solve_3idx'.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0
    infinity = solver.infinity()

    # Variables
    x = {}
    u = {}
    for i in range(n_nodes):
        u[i] = solver.NumVar(0, n_nodes, f'u_{i}')
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # Constraints (Mutable)
    cons_out = {}
    cons_in = {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'out_{i}')
        c_in = solver.Constraint(0, 0, f'in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    # MTZ Constraints (Static)
    # u_i - u_j + N*x_ij <= N - 1
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
                c.SetCoefficient(u[i], 1)
                c.SetCoefficient(u[j], -1)
                c.SetCoefficient(x[(i, j)], n_nodes)

    # Objective
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    @lru_cache(maxsize=10000)
    def _solve_3idx(active_tuple):
        # Tuple structure: (current_node, sorted_unvisited...)
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0) # Depot

        max_u = len(active_set) # Max position in path

        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    # Start: Out=1, In=0, u=0
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(0, 0)
                    u[i].SetBounds(0, 0)
                elif i == 0:
                    # End: Out=0, In=1, u=max
                    cons_out[i].SetBounds(0, 0)
                    cons_in[i].SetBounds(1, 1)
                    u[i].SetBounds(1, max_u)
                else:
                    # Mid: Out=1, In=1, u in [1, max]
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(1, 1)
                    u[i].SetBounds(1, max_u)
            else:
                # Inactive
                cons_out[i].SetBounds(0, 0)
                cons_in[i].SetBounds(0, 0)
                u[i].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # --- WRAPPER ---
    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_var]
        curr = state[location_var]
        if not unvisited and curr == 0: return 0.0
        
        # Key must distinguish current node (start of path)
        key = (curr,) + tuple(sorted(list(unvisited)))
        return _solve_3idx(key)

    return h_lp_relaxation_3_idx


def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    """
    2. Persistent 2-Index TSP Relaxation (Assignment/Flow Formulation).
    - Variables: x[i,j] (Flow). NO MTZ variables.
    - Logic: Relaxed connectivity (Subtours allowed). Faster than 3-Index.
    - Pattern: Cached internal worker '_solve_2idx'.
    """
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # --- INITIALIZATION ---
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0

    # Variables
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    # Constraints (Degree/Flow)
    cons_out = {}
    cons_in = {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'out_{i}')
        c_in = solver.Constraint(0, 0, f'in_{i}')
        for j in range(n_nodes):
            if i != j:
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    # Objective
    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    # --- CACHED WORKER ---
    @lru_cache(maxsize=10000)
    def _solve_2idx(active_tuple):
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node)
        active_set.add(0)

        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    # Start: Out=1, In=0
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(0, 0)
                elif i == 0:
                    # End: Out=0, In=1
                    cons_out[i].SetBounds(0, 0)
                    cons_in[i].SetBounds(1, 1)
                else:
                    # Mid: Out=1, In=1
                    cons_out[i].SetBounds(1, 1)
                    cons_in[i].SetBounds(1, 1)
            else:
                # Inactive
                cons_out[i].SetBounds(0, 0)
                cons_in[i].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
            return float(objective.Value())
        return 0.0

    # --- WRAPPER ---
    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        curr = state[location_var]
        if not unvisited and curr == 0: return 0.0
        
        key = (curr,) + tuple(sorted(list(unvisited)))
        return _solve_2idx(key)

    return h_lp_relaxation_2_idx

In [6]:
def dual_bound_expression_function(didp_bundle):
    """ 
    Registry containing ALL heuristics (Combinatorial + LP) for TSP.
    """
    model, metadata = didp_bundle
    
    # Extract metadata
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list)
    num_nodes = metadata['num_nodes']

    # Pre-computation
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    min_outgoing_arr = np.min(masked_cost, axis=1)
    min_incoming_arr = np.min(masked_cost, axis=0)

    # --- Initialize LP Bounds ---
    h_lp_relaxation_3_idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata)
    h_lp_relaxation_2_idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata)

    # ==========================================
    # COMBINATORIAL BOUNDS (Internal Workers)
    # ==========================================

    # --- Degree Average Bound ---
    @lru_cache(maxsize=100000)
    def _calc_degree(active_tuple):
        nodes = list(active_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)
        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        # Current (0) excludes Incoming, Depot (-1) excludes Outgoing
        sum_in = np.sum(mins_in[1:])
        sum_out = np.sum(mins_out[:-1])
        return float(0.5 * (sum_in + sum_out))

    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        # Tuple: (current, U..., depot)
        active_list = [curr] + sorted(list(U))
        if 0 not in active_list: active_list.append(0)
        return _calc_degree(tuple(active_list))

    # --- Global Min Flow ---
    @lru_cache(maxsize=100000)
    def _calc_min_flow_static(unvisited_tuple):
        val_out = sum(min_outgoing_arr[u] for u in unvisited_tuple)
        val_in = sum(min_incoming_arr[u] for u in unvisited_tuple)
        return val_out, val_in

    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        if not U and curr == 0: return 0.0
        
        val_out, val_in = _calc_min_flow_static(tuple(sorted(list(U))))
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            val_in += min_incoming_arr[0]
        return float(max(val_out, val_in))

    # --- MST Bound ---
    @lru_cache(maxsize=100000)
    def _calc_mst(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_mst(state):
        U = state[unvisited_var]
        return _calc_mst(tuple(sorted(list(U))))

    # --- 1-Tree Bound ---
    @lru_cache(maxsize=100000)
    def _calc_1tree(unvisited_tuple):
        subset = list(unvisited_tuple)
        depot_edges = sorted(cost_matrix[0, subset])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        if len(subset) > 1:
            sub_mat = cost_matrix[np.ix_(subset, subset)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0
        return float(mst_val + e1 + e2)

    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_1tree(tuple(sorted(list(U))))

    # --- Assignment Bound ---
    @lru_cache(maxsize=100000)
    def _calc_assignment(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row, col = linear_sum_assignment(assign_mat)
        return float(assign_mat[row, col].sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_assignment(tuple(sorted(list(U))))

    # --- Eigenvalue Bound ---
    @lru_cache(maxsize=100000)
    def _calc_eigen(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        N = len(nodes)
        if N < 2: return 0.0
        
        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])
        
        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals): phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
        return float(phi)

    def h_eigen(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_eigen(tuple(sorted(list(U))))

    # Return valid registry
    return automatic_creation_of_dual_bounds_registry(locals())

# Execution Line
dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_lp_relaxation_3_idx': <function __main__.create_persistent_lp_relaxation_3_index_dual_bounds.<locals>.h_lp_relaxation_3_idx(state)>,
 'h_lp_relaxation_2_idx': <function __main__.create_persistent_lp_relaxation_2_index_dual_bounds.<locals>.h_lp_relaxation_2_idx(state)>,
 'h_degree_average': <function __main__.dual_bound_expression_function.<locals>.h_degree_average(state)>,
 'h_global_min_flow': <function __main__.dual_bound_expression_function.<locals>.h_global_min_flow(state)>,
 'h_mst': <function __main__.dual_bound_expression_function.<locals>.h_mst(state)>,
 'h_1tree': <function __main__.dual_bound_expression_function.<locals>.h_1tree(state)>,
 'h_assignment': <function __main__.dual_bound_expression_function.<locals>.h_assignment(state)>,
 'h_eigen': <function __main__.dual_bound_expression_function.<locals>.h_eigen(state)>}

# **Hyperparamters setting**

In [7]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 15      # Size of the population in each generation
GENERATIONS = 10          # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.02
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
# OPTIMAL_COST_REFERENCE= 400
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 10 #seconds

# **Execution**

In [ ]:
# --- EXECUTE CLEANUP ---
# Run this right before your main loop
clean_batch_data(OUTPUT_CSV, LOGS_DIR)
df_input = pd.read_csv(INPUT_CSV)
processed = get_processed_instances(OUTPUT_CSV, LOGS_DIR)

if len(processed) < len(df_input):
    print(f"🚀 Ready to start batch Run. {len(processed)}/{len(df_input)} instances already completed.")
    print(f"🚀 Starting batch Run. {len(processed)}/{len(df_input)} instances already completed.")
else:
    print(f"🚀 {len(processed)}/{len(df_input)} instances already completed. No need further running")
    
# Modified check in Cell 4
for index, row in df_input.iterrows():
    instance_name = row['Instance']
    log_file_path = os.path.join(LOGS_DIR, f"{instance_name}_log.txt")
    
    # Check BOTH the CSV record AND the actual log file
    if instance_name in processed and os.path.exists(log_file_path):
        continue # Safe to skip
    
    optimal_singal = row['Is Optimal']
    if optimal_singal == True:
        optimal_cost = row['Cost']
    else:
        optimal_cost = row['LP Relaxed LB']
    file_path = os.path.join(DATA_DIR, instance_name)
    print(f"\nProcessing {instance_name} with Ref Cost: {optimal_cost} (Optimality ?: {optimal_singal}) ")

    try:
        # A. Update Global Data for this instance
        global current_num_locations, current_travel_cost
        current_num_locations, current_travel_cost = read_tsp_cappart_format(file_path)
        
        # B. Configure Params
        params = EAHyperparameters(
            # --- 1. Population ---
            population_size=POPULATION_SIZE,          
            generations=GENERATIONS,
            crossover_rate=CROSSOVER_RATE,
            mutation_rate=MUTATION_RATE,
            elitism_rate=ELITISM_RATE,           

            # --- 2. Ranges & Constraints ---
            lb_range_of_constant=LB_range_of_constant,
            ub_range_of_constant=UB_range_of_constant,
            min_chromosome_length=min_chromosome_length,     
            max_chromosome_length=max_chromosome_length,   

            # --- 3. Operator Specifics ---
            tournament_size=tournament_size,                             
            tournament_probability=tournament_probability,                    
            mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
            homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
            subtree_crossover_probability=subtree_crossover_probability,             
            uniform_crossover_probability=uniform_crossover_probability,             

            # --- 4. Problem Specific ---
            reference_point=optimal_cost,         
            solver_time_limit=SOLVER_TIME_LIMIT,
            
            # Optional: You can override available operations if needed
            available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
        )

        # C. Run EA (Redirecting output to file to keep notebook clean)
        log_file = os.path.join(LOGS_DIR, f"{instance_name}_log.txt")
        start_time = time.time()
        
        # Capture print outputs to log file
        with open(log_file, "w", encoding="utf-8") as f:
            with contextlib.redirect_stdout(f):
                didp_model = creation_of_didp_model_function()
                dual_bound_funcs = dual_bound_expression_function(didp_model)
                best_ind = evolution_algorithm_execution(
                    didp_model_registry=creation_of_didp_model_function,
                    dual_bound_expression_function=dual_bound_expression_function,
                    params=params
                )
                # --- NEW CODE: Print best_ind here to save it to the log ---
                print("\n" + "="*40)
                print("FINAL BEST INDIVIDUAL")
                print("="*40)
                print(best_ind) 
                # -----------------------------------------------------------
        
        total_time = time.time() - start_time

        # D. Save Results
        result_data = {
            "Instance": instance_name,
            "Total_Time_(s)": round(total_time, 2),
            "Best_Fitness": best_ind['fitness'],
            "Best_Chromosome": str(best_ind['chromosome']),
            "Log_File": log_file
        }
        append_result_to_csv(result_data, OUTPUT_CSV)
        print(f"   ✅ Finished! Best Fit: {best_ind['fitness']:.4f} | Time: {total_time:.2f}s")

    except Exception as e:
        print(f"   ❌ Failed: {e}")
    
    finally:
        # E. Cleanup Memory
        gc.collect()

print("\n🎉 Batch Run Complete!")

🧹 Starting Data Cleanup...
   -> CSV saved. Current number of row is: 15 (was 15).
   -> Deleted orphan log: 4.txt_log.txt (Not in CSV).
✨ Data Cleanup Complete.

🚀 Ready to start batch Run. 15/20 instances already completed.
🚀 Starting batch Run. 15/20 instances already completed.

Processing 4.txt with Ref Cost: 486.3200000000001 (Optimality ?: False (Time Limit)) 
